# DEG analysis — pseudobulk comparison across injury models

Differentially expressed genes (DEG) between **inj_1** and **inj_2** injury models,
stratified by **phase** (early_acute · subacute · late).

**Input:** annotated spatial `.h5ad` with:
- `adata.obs["celltype"]` — broad cell type labels (from Tangram or scVI annotation)
- `adata.obs["meta_domain"]` — spatial meta-domain ID (from BANKSY correlation analysis)
- `adata.obs["inj_type"]` — injury type (`inj_1` / `inj_2`)
- `adata.obs["day"]` — timepoint (`d1`, `d2`, ..., `d14`)
- `adata.obs["batch"]` — slide/section ID
- `adata.layers["counts"]` — raw integer counts (required for pseudobulk)

**Three statistical methods are provided** (choose one):
- `Wilcoxon` via `scanpy.tl.rank_genes_groups` (per-cell test, fastest, no pseudobulk needed — but treats each cell as independent, inflating significance for highly sampled conditions)
- `edgeR` via `Rscript` subprocess (recommended for publication — pseudobulk, more robust for spatial data)
- `DESeq2` via `pydeseq2` (Python-native pseudobulk alternative, no R required)

Wilcoxon is useful for a **quick first look** at candidate genes, since it requires no
replicate pooling and runs directly on cells. However it does not account for
pseudoreplication (cells from the same animal are not independent samples), so
p-values are typically far more liberal than edgeR/DESeq2. Use edgeR or DESeq2 for
any result you intend to report.

**How samples are grouped for DEG:**
Individual slides have too few cells per cell type to run DEG directly.
Instead, slides are **pooled into biological replicates** by grouping
nearby timepoints together (e.g. `inj_1_d2_1 + inj_1_d2_2 + inj_1_d3_1 → rep1`).
Each replicate aggregates raw counts across all cells of the target cell type
in the target meta-domain — this is the **pseudobulk** approach.

**Bias correction strategy:**
MERSCOPE data can have variable transcript detection efficiency across slides.
This creates a confound if one injury type systematically has higher counts.
We measure this as the point-biserial correlation between injury type and
mean counts per replicate:
- If |r| ≤ 0.75 and enough replicates exist: include `mean_counts_scaled` as a covariate in the model
- If |r| > 0.75 (too collinear): omit the covariate — it would absorb the injury signal

**Outputs per phase:**
- DEG table (CSV): `log2FoldChange`, `pvalue`, `padj`
- Volcano plot (PDF)
- Heatmap of significant DEGs (PDF)


## 1 · Imports

In [ ]:
import os
import subprocess
import tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import scanpy as sc
from scipy.sparse import issparse

## 2 · Settings

Edit this cell to configure the analysis.

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
INPUT_H5AD = "/path/to/your/annotated_spatial.h5ad"
OUT_DIR    = "/path/to/your/output/DEG/"
RSCRIPT    = "/usr/local/bin/Rscript"   # path to Rscript binary

os.makedirs(OUT_DIR, exist_ok=True)

# ── Column names in adata.obs ──────────────────────────────────────────────────
CELLTYPE_COL   = "celltype"     # e.g. "Fibroblasts", "Immune_cells"
METADOMAIN_COL = "meta_domain"  # spatial domain ID
INJURY_COL     = "inj_type"     # "inj_1" or "inj_2"
TIME_COL       = "day"          # "d1", "d2", ..., "d14"
SAMPLE_COL     = "batch"        # slide/section ID

# ── Analysis identity ──────────────────────────────────────────────────────────
# These labels appear in all output filenames and plot titles.
# Change both when switching cell type or domain.
CELLTYPE_VALUE = "Fibroblasts"   # must match adata.obs[CELLTYPE_COL]
METADOMAIN_VALUE = "6"           # must match adata.obs[METADOMAIN_COL]
ANALYSIS_LABEL = "fibro_metadomain6"

# ── Phase grouping ────────────────────────────────────────────────────────────
# Days are collapsed into three phases to increase statistical power.
# Adjust the mapping if your timepoints differ.
PHASE_MAP = {
    "d1": "early_acute", "d2": "early_acute", "d3": "early_acute",
    "d4": "subacute",    "d5": "subacute",    "d7": "subacute",
    "d9": "late",        "d14": "late",
}
PHASE_ORDER = ["early_acute", "subacute", "late"]

# ── Replicate pooling ─────────────────────────────────────────────────────────
# Map individual slide IDs (batch) → pooled replicate IDs.
# Slides from nearby timepoints with the same biological replicate
# (e.g. same animal) are grouped together to create pseudo-replicates
# with enough cells for pseudobulk DEG.
# Add or remove entries based on your experimental design.
REPLICATE_MAP = {
    # early_acute · inj_1
    "inj_1_d3_4": "rep1", "inj_1_d3_2": "rep2", "inj_1_d3_3": "rep3",
    "inj_1_d3_1": "rep4", "inj_1_d2_2": "rep5", "inj_1_d2_1": "rep6",
    # early_acute · inj_2
    "inj_2_d1_7": "rep41", "inj_2_d1_6": "rep41",
    "inj_2_d2_4": "rep41", "inj_2_d2_5": "rep41", "inj_2_d3_2": "rep41",
    "inj_2_d2_7": "rep7",  "inj_2_d3_4": "rep7",
    "inj_2_d1_9": "rep7",  "inj_2_d1_8": "rep7",
    "inj_2_d2_6": "rep8",
    # subacute · inj_1
    "inj_1_d7_3": "rep9",  "inj_1_d7_2": "rep10", "inj_1_d5_8": "rep11",
    "inj_1_d4_1": "rep12", "inj_1_d5_9": "rep13", "inj_1_d4_2": "rep14",
    "inj_1_d5_7": "rep15", "inj_1_d4_3": "rep16", "inj_1_d7_1": "rep17",
    # subacute · inj_2
    "inj_2_d4_8": "rep18", "inj_2_d4_7": "rep18",
    "inj_2_d4_9": "rep19", "inj_2_d4_6": "rep19",
    "inj_2_d4_5": "rep20", "inj_2_d4_2": "rep20",
    "inj_2_d4_3": "rep21", "inj_2_d4_1": "rep21",
    "inj_2_d5_6": "rep22", "inj_2_d5_7": "rep22",
    "inj_2_d5_1": "rep23", "inj_2_d7_13": "rep24",
    "inj_2_d7_14": "rep25","inj_2_d5_5": "rep26",
    "inj_2_d5_2": "rep27", "inj_2_d7_10": "rep28",
    # late · inj_1
    "inj_1_d9_2":  "rep29", "inj_1_d14_1": "rep30", "inj_1_d14_6": "rep31",
    "inj_1_d14_5": "rep32", "inj_1_d14_4": "rep33", "inj_1_d14_3": "rep34",
    "inj_1_d14_2": "rep35", "inj_1_d9_1":  "rep36",
    # late · inj_2
    "inj_2_d9_6": "rep37", "inj_2_d9_7": "rep37",
    "inj_2_d9_8": "rep38", "inj_2_d9_5": "rep39", "inj_2_d9_9": "rep40",
}

# ── Bias correction thresholds ────────────────────────────────────────────────
# If abs(correlation between injury and mean_counts) > this threshold,
# the covariate is too collinear and is dropped from the design.
COVARIATE_R_THRESHOLD = 0.75
MIN_REPS_PER_GROUP_FOR_COVARIATE = 3  # minimum replicates per group to safely add covariate
MIN_CELLS_PER_REPLICATE = 10          # replicates below this are dropped before fitting

# ── DEG thresholds ─────────────────────────────────────────────────────────────
PVALUE_THRESH = 0.05
LFC_THRESH    = 0.58   # log2(1.5) — minimum fold change to call significant
TOP_N_GENES   = 500    # number of genes to export per group

## 3 · Load data

In [ ]:
adata = sc.read_h5ad(INPUT_H5AD)

# Verify required columns exist
for col in [CELLTYPE_COL, METADOMAIN_COL, INJURY_COL, TIME_COL, SAMPLE_COL]:
    if col not in adata.obs.columns:
        raise KeyError(f"Missing adata.obs['{col}']")
if "counts" not in adata.layers:
    raise KeyError("adata.layers['counts'] not found — raw integer counts required for pseudobulk DEG")

print(f"Loaded: {adata.shape[0]:,} cells x {adata.shape[1]:,} genes")
print(f"Injury types: {adata.obs[INJURY_COL].value_counts().to_dict()}")
print(f"Cell types: {adata.obs[CELLTYPE_COL].value_counts().to_dict()}")

## 4 · Subset and prepare

In [ ]:
# Subset to target cell type, meta-domain, and injuries of interest
adata_deg = adata[
    (adata.obs[CELLTYPE_COL] == CELLTYPE_VALUE) &
    (adata.obs[METADOMAIN_COL] == METADOMAIN_VALUE) &
    (adata.obs[INJURY_COL].isin(["inj_1", "inj_2"]))
].copy()

print(f"Cells after subsetting: {adata_deg.n_obs:,}")
if adata_deg.n_obs == 0:
    raise ValueError("No cells found — check CELLTYPE_VALUE, METADOMAIN_VALUE, and INJURY_COL values")

# Assign phase
adata_deg.obs["phase"] = adata_deg.obs[TIME_COL].map(PHASE_MAP)
n_before = adata_deg.n_obs
adata_deg = adata_deg[adata_deg.obs["phase"].notna()].copy()
print(f"Cells with valid phase: {adata_deg.n_obs:,}  (dropped {n_before - adata_deg.n_obs})")

# Map slides to pooled replicates
adata_deg.obs["replicate"] = adata_deg.obs[SAMPLE_COL].map(REPLICATE_MAP)
unmapped = adata_deg.obs.loc[adata_deg.obs["replicate"].isna(), SAMPLE_COL].unique()
if len(unmapped) > 0:
    print(f"\n⚠ Unmapped slides (will be dropped): {list(unmapped)}")
    adata_deg = adata_deg[adata_deg.obs["replicate"].notna()].copy()

print(f"Unique replicates: {adata_deg.obs['replicate'].nunique()}")

# Compute total counts per cell (used for bias assessment)
adata_deg.obs["total_counts"] = np.asarray(
    adata_deg.layers["counts"].sum(axis=1)
).ravel()

## 5 · QC — transcript detection bias

Check whether one injury type has systematically higher counts per cell.
This can confound DEG if not corrected.

In [ ]:
qc_summary = (
    adata_deg.obs
    .groupby(["phase", INJURY_COL])["total_counts"]
    .agg(["count", "mean", "median", "std"])
    .rename(columns={"count": "n_cells"})
    .round(2)
)
print("Transcripts per cell QC:")
print(qc_summary.to_string())

fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=True)
for ax, phase in zip(axes, PHASE_ORDER):
    sub = adata_deg.obs[adata_deg.obs["phase"] == phase]
    sns.boxplot(data=sub, x=INJURY_COL, y="total_counts",
                palette={"inj_1": "steelblue", "inj_2": "orange"},
                order=["inj_1", "inj_2"], ax=ax, showfliers=False)
    sns.stripplot(data=sub, x=INJURY_COL, y="total_counts",
                  palette={"inj_1": "steelblue", "inj_2": "orange"},
                  order=["inj_1", "inj_2"], ax=ax, size=2, alpha=0.3, jitter=True)
    ax.set_title(phase)
    ax.set_xlabel("")
plt.suptitle(f"Transcripts per cell — {ANALYSIS_LABEL}")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}QC_transcripts_{ANALYSIS_LABEL}.pdf", bbox_inches="tight")
plt.show()

## 6 · Pseudobulk aggregation

Sum raw counts across all cells within each replicate.
This creates one count profile per replicate — the input for edgeR/DESeq2.

In [ ]:
def make_pseudobulk(adata, rep_col):
    """Sum raw counts per replicate and return (counts_df, metadata_df)."""
    counts_mat = adata.layers["counts"]
    if issparse(counts_mat):
        counts_mat = counts_mat.tocsr()

    replicates  = adata.obs[rep_col].astype(str).values
    unique_reps = pd.Index(pd.unique(replicates))

    pb_rows = []
    for r in unique_reps:
        idx    = np.where(replicates == r)[0]
        summed = counts_mat[idx].sum(axis=0)
        pb_rows.append(np.asarray(summed).ravel())

    pb_counts = pd.DataFrame(pb_rows, index=unique_reps,
                              columns=adata.var_names, dtype=int)

    pb_meta = (
        adata.obs.groupby(rep_col, observed=True)
        [[INJURY_COL, "phase", TIME_COL]]
        .agg("first").copy()
    )
    pb_meta["n_cells"]     = adata.obs.groupby(rep_col).size()
    pb_meta["mean_counts"] = adata.obs.groupby(rep_col)["total_counts"].mean().round(2)

    return pb_counts, pb_meta


pb_counts_list, pb_meta_list = [], []
for phase in PHASE_ORDER:
    src_phase = adata_deg[adata_deg.obs["phase"] == phase].copy()
    pc, pm = make_pseudobulk(src_phase, "replicate")
    pb_counts_list.append(pc)
    pb_meta_list.append(pm)

pb_counts = pd.concat(pb_counts_list).fillna(0).astype(int)
pb_meta   = pd.concat(pb_meta_list)

print("Replicates per phase & injury:")
print(
    pb_meta.groupby(["phase", INJURY_COL])["n_cells"]
    .agg(["count", "sum", "min", "max"])
    .rename(columns={"count": "n_reps", "sum": "total_cells",
                     "min": "min_cells", "max": "max_cells"})
    .to_string()
)

# Save replicate summary
summary = (
    pb_meta.reset_index().rename(columns={"replicate": "replicate"})
    [["phase", INJURY_COL, "replicate", "n_cells", "mean_counts"]]
)
summary["phase"] = pd.Categorical(summary["phase"], categories=PHASE_ORDER, ordered=True)
summary.sort_values(["phase", INJURY_COL]).to_csv(
    f"{OUT_DIR}replicates_summary_{ANALYSIS_LABEL}.csv", index=False
)

## 7 · Automatic bias correction decision

For each phase, measure the correlation between injury type and mean counts per replicate.
A high correlation means the covariate is collinear with the variable of interest —
in that case we drop it to avoid absorbing the biological signal.

In [ ]:
def decide_covariate(meta_ph: pd.DataFrame, phase_name: str) -> dict:
    """
    Decide whether to include mean_counts_scaled covariate for a phase.

    Logic:
      - require >= MIN_REPS_PER_GROUP_FOR_COVARIATE replicates in each group
      - require abs(point-biserial r) <= COVARIATE_R_THRESHOLD
    If both conditions are met: include covariate. Otherwise: injury-only model.
    """
    n_inj_1  = int((meta_ph[INJURY_COL] == "inj_1").sum())
    n_inj_2 = int((meta_ph[INJURY_COL] == "inj_2").sum())

    if meta_ph["mean_counts"].nunique() <= 1:
        r, p = np.nan, np.nan
    else:
        injury_binary = (meta_ph[INJURY_COL] == "inj_1").astype(int)
        r, p = stats.pointbiserialr(injury_binary, meta_ph["mean_counts"])

    enough_reps   = (n_inj_1 >= MIN_REPS_PER_GROUP_FOR_COVARIATE and
                     n_inj_2 >= MIN_REPS_PER_GROUP_FOR_COVARIATE)
    low_collinear = pd.notna(r) and abs(r) <= COVARIATE_R_THRESHOLD
    use_cov = bool(enough_reps and low_collinear)

    if not enough_reps:
        reason = f"too few replicates ({n_inj_1} inj_1, {n_inj_2} inj_2)"
    elif pd.isna(r):
        reason = "mean_counts has no variation"
    elif abs(r) > COVARIATE_R_THRESHOLD:
        reason = f"collinearity too high (|r|={abs(r):.3f} > {COVARIATE_R_THRESHOLD})"
    else:
        reason = f"acceptable collinearity (|r|={abs(r):.3f} ≤ {COVARIATE_R_THRESHOLD})"

    print(f"[{phase_name}] r={r:.3f}  p={p:.3f}  →  "
          f"{'USE covariate' if use_cov else 'NO covariate'}  ({reason})")

    return {"phase": phase_name, "n_inj_1_reps": n_inj_1, "n_inj_2_reps": n_inj_2,
            "r": r, "p": p, "use_covariate": use_cov, "reason": reason}


cov_decisions = []
for phase in PHASE_ORDER:
    meta_ph = pb_meta[pb_meta["phase"] == phase].copy()
    cov_decisions.append(decide_covariate(meta_ph, phase))

cov_df = pd.DataFrame(cov_decisions)
cov_df.to_csv(f"{OUT_DIR}covariate_decision_{ANALYSIS_LABEL}.csv", index=False)
print()
print(cov_df[["phase", "r", "use_covariate", "reason"]].to_string(index=False))

## 8 · DEG — choose your method

Three approaches are available. **edgeR is recommended for any reported result.**
Wilcoxon is fast and useful for a quick look but does not correct for pseudoreplication.

Use `edgeR` or `DESeq2` (pseudobulk methods) for publication-quality results, and
`Wilcoxon` for rapid exploration only.

In [ ]:
# Wilcoxon rank-sum test on individual cells (scanpy).
# Fast and simple — no pseudobulk/replicate pooling required.
# Run on the same subset (cell type + meta-domain) used for pseudobulk, per phase.

def run_wilcoxon(adata_subset_obj, phase_name: str, layer: str = "transformed") -> pd.DataFrame:
    """
    Wilcoxon DEG for one phase: inj_1 vs inj_2, using cells directly (no pooling).
    Positive log2FoldChange (scanpy's 'logfoldchanges') = higher in inj_1.
    """
    ad_phase = adata_subset_obj[adata_subset_obj.obs["phase"] == phase_name].copy()

    if not ({"inj_1", "inj_2"} <= set(ad_phase.obs[INJURY_COL].unique())):
        raise ValueError(f"Phase '{phase_name}': missing a condition")

    # use the requested layer (must be log-normalized for meaningful fold changes)
    if layer not in ad_phase.layers:
        raise KeyError(f"layer '{layer}' not found in adata.layers — "
                       f"available: {list(ad_phase.layers.keys())}")
    ad_phase.X = ad_phase.layers[layer].copy()

    sc.tl.rank_genes_groups(
        ad_phase,
        groupby=INJURY_COL,
        groups=["inj_1"],
        reference="inj_2",
        method="wilcoxon",
        use_raw=False,
    )

    res = sc.get.rank_genes_groups_df(ad_phase, group="inj_1")
    res = res.rename(columns={
        "names": "gene",
        "logfoldchanges": "log2FoldChange",
        "pvals": "pvalue",
        "pvals_adj": "padj",
    }).set_index("gene")
    res["n_cells_inj_1"]  = (ad_phase.obs[INJURY_COL] == "inj_1").sum()
    res["n_cells_inj_2"] = (ad_phase.obs[INJURY_COL] == "inj_2").sum()

    return res.sort_values("pvalue")


results_wilcoxon = {}
for phase in PHASE_ORDER:
    try:
        res = run_wilcoxon(adata_deg, phase)
        results_wilcoxon[phase] = res
        res.to_csv(f"{OUT_DIR}DEG_{ANALYSIS_LABEL}_{phase}_inj_1_vs_inj_2_wilcoxon.csv")
        print(f"  → saved: {phase}  "
              f"(n_inj_1={res['n_cells_inj_1'].iloc[0]}, n_inj_2={res['n_cells_inj_2'].iloc[0]})")
    except (ValueError, KeyError) as e:
        print(f"  ⚠ Skipping {phase}: {e}")

### Option B — edgeR (recommended for publication)

In [ ]:
R_SCRIPT_TEMPLATE = r"""
suppressPackageStartupMessages(library(edgeR))

counts_path   <- "{counts_path}"
meta_path     <- "{meta_path}"
out_path      <- "{out_path}"
use_covariate <- {use_covariate}

counts_df <- read.csv(counts_path, row.names=1, check.names=FALSE)
counts    <- t(as.matrix(counts_df))      # genes x samples for edgeR

meta   <- read.csv(meta_path, row.names=1, check.names=FALSE)
meta   <- meta[colnames(counts), , drop=FALSE]

injury <- factor(meta$inj_type, levels=c("inj_2", "inj_1"))  # inj_2 = reference

if (use_covariate) {{
    mc_scaled <- meta$mean_counts_scaled
    design    <- model.matrix(~ mc_scaled + injury)
}} else {{
    design    <- model.matrix(~ injury)
}}
rownames(design) <- colnames(counts)

dge  <- DGEList(counts=counts, group=injury)
keep <- rowSums(dge$counts) >= 5           # relaxed pre-filter
dge  <- dge[keep, , keep.lib.sizes=FALSE]
dge  <- calcNormFactors(dge, method="TMM")

dge  <- estimateDisp(dge, design, robust=TRUE)
fit  <- glmQLFit(dge, design, robust=TRUE)
qlf  <- glmQLFTest(fit, coef=ncol(design)) # last coef = injury

result <- topTags(qlf, n=Inf, sort.by="PValue")$table
colnames(result)[colnames(result)=="logFC"]  <- "log2FoldChange"
colnames(result)[colnames(result)=="PValue"] <- "pvalue"
colnames(result)[colnames(result)=="FDR"]    <- "padj"
result$gene <- rownames(result)
result <- result[, c("gene", "log2FoldChange", "logCPM", "F", "pvalue", "padj")]

write.csv(result, out_path, row.names=FALSE)
cat("edgeR done:", nrow(result), "genes tested\n")
"""


def run_edgeR(phase_name: str) -> pd.DataFrame:
    """
    Run edgeR for one phase.
    Positive log2FoldChange = higher in inj_1.
    Covariate use is decided automatically per decide_covariate().
    """
    meta_ph   = pb_meta[pb_meta["phase"] == phase_name].copy()
    counts_ph = pb_counts.loc[meta_ph.index].copy()

    # drop under-populated replicates
    valid = meta_ph["n_cells"] >= MIN_CELLS_PER_REPLICATE
    if (~valid).any():
        print(f"  [{phase_name}] dropping {(~valid).sum()} replicate(s) "
              f"with <{MIN_CELLS_PER_REPLICATE} cells")
    meta_ph   = meta_ph[valid]
    counts_ph = counts_ph.loc[meta_ph.index]

    if not ({"inj_1", "inj_2"} <= set(meta_ph[INJURY_COL].unique())):
        raise ValueError(f"Phase '{phase_name}': missing a condition after filtering")

    # Python pre-filter: detected in at least 2 replicates
    keep = (counts_ph > 0).sum(axis=0) >= 2
    counts_ph = counts_ph.loc[:, keep]

    use_cov = cov_df.set_index("phase").loc[phase_name, "use_covariate"]
    meta_ph["mean_counts_scaled"] = meta_ph["mean_counts"] / meta_ph["mean_counts"].mean()

    with tempfile.TemporaryDirectory() as tmpdir:
        counts_path = os.path.join(tmpdir, "counts.csv")
        meta_path   = os.path.join(tmpdir, "meta.csv")
        out_path    = os.path.join(tmpdir, "result.csv")
        r_path      = os.path.join(tmpdir, "run.R")

        counts_ph.to_csv(counts_path)
        meta_ph[[INJURY_COL, "mean_counts", "mean_counts_scaled"]].to_csv(meta_path)

        with open(r_path, "w") as f:
            f.write(R_SCRIPT_TEMPLATE.format(
                counts_path=counts_path,
                meta_path=meta_path,
                out_path=out_path,
                use_covariate="TRUE" if use_cov else "FALSE",
            ))

        proc = subprocess.run([RSCRIPT, "--vanilla", r_path],
                              capture_output=True, text=True)
        if proc.returncode != 0:
            print(f"R error:\n{proc.stderr}")
            raise RuntimeError(f"edgeR failed for phase '{phase_name}'")
        if proc.stdout.strip():
            print(f"  {proc.stdout.strip()}")

        res = pd.read_csv(out_path, index_col="gene")

    return res.sort_values("pvalue")


results_edgeR = {}
for phase in PHASE_ORDER:
    try:
        res = run_edgeR(phase)
        results_edgeR[phase] = res
        res.to_csv(f"{OUT_DIR}DEG_{ANALYSIS_LABEL}_{phase}_inj_1_vs_inj_2_edgeR.csv")
        print(f"  → saved: {phase}")
    except (ValueError, RuntimeError) as e:
        print(f"  ⚠ Skipping {phase}: {e}")

### Option C — DESeq2 (Python-native, no R required)

Recommended when edgeR is not available or you prefer a Python-only workflow.
Note that DESeq2 is less robust when there are very few replicates per group.

In [ ]:
# Uncomment to use DESeq2 instead of edgeR
# Requires: pip install pydeseq2

# from pydeseq2.dds import DeseqDataSet
# from pydeseq2.ds import DeseqStats
# from pydeseq2.default_inference import DefaultInference
# import sys
#
# def run_deseq2(phase_name: str) -> pd.DataFrame:
#     meta_ph   = pb_meta[pb_meta["phase"] == phase_name].copy()
#     counts_ph = pb_counts.loc[meta_ph.index].copy()
#
#     valid = meta_ph["n_cells"] >= MIN_CELLS_PER_REPLICATE
#     meta_ph   = meta_ph[valid]
#     counts_ph = counts_ph.loc[meta_ph.index]
#
#     keep = (counts_ph > 0).sum(axis=0) >= 2
#     counts_ph = counts_ph.loc[:, keep]
#
#     use_cov = cov_df.set_index("phase").loc[phase_name, "use_covariate"]
#
#     meta_ph[INJURY_COL] = pd.Categorical(meta_ph[INJURY_COL],
#                                          categories=["inj_2", "inj_1"])
#     if use_cov:
#         meta_ph["mean_counts_scaled"] = (
#             meta_ph["mean_counts"] / meta_ph["mean_counts"].mean()
#         )
#         design_factors = [INJURY_COL, "mean_counts_scaled"]
#     else:
#         design_factors = INJURY_COL
#
#     inference = DefaultInference(n_cpus=1)
#     with open(os.devnull, "w") as devnull:
#         old_stdout = sys.stdout
#         sys.stdout = devnull
#         try:
#             dds = DeseqDataSet(counts=counts_ph, metadata=meta_ph,
#                                design_factors=design_factors,
#                                refit_cooks=True, inference=inference)
#             dds.deseq2()
#             stat = DeseqStats(dds, contrast=[INJURY_COL, "inj_1", "inj_2"],
#                               inference=inference)
#             stat.summary()
#         finally:
#             sys.stdout = old_stdout
#     return stat.results_df.sort_values("pvalue")
#
# results_deseq2 = {}
# for phase in PHASE_ORDER:
#     try:
#         res = run_deseq2(phase)
#         results_deseq2[phase] = res
#         res.to_csv(f"{OUT_DIR}DEG_{ANALYSIS_LABEL}_{phase}_inj_1_vs_inj_2_deseq2.csv")
#     except (ValueError, RuntimeError) as e:
#         print(f"  ⚠ Skipping {phase}: {e}")

## 9 · Visualisation

### Volcano plots

In [ ]:
def volcano(df, phase_name, lfc_thresh=LFC_THRESH, pval_thresh=PVALUE_THRESH, top_n=10,
            save_path=None):
    df = df.dropna(subset=["log2FoldChange", "pvalue"]).copy()
    df["neglog10p"] = -np.log10(df["pvalue"].clip(lower=1e-300))

    sig_inj_1  = (df["pvalue"] < pval_thresh) & (df["log2FoldChange"] >  lfc_thresh)
    sig_inj_2 = (df["pvalue"] < pval_thresh) & (df["log2FoldChange"] < -lfc_thresh)
    other      = ~(sig_inj_1 | sig_inj_2)

    use_cov = cov_df.set_index("phase").loc[phase_name, "use_covariate"]
    note = "covariate corrected" if use_cov else "injury only"

    fig, ax = plt.subplots(figsize=(9, 7))
    ax.scatter(df.loc[other,      "log2FoldChange"], df.loc[other,      "neglog10p"],
               s=10, alpha=0.35, color="grey",      label="Not significant")
    ax.scatter(df.loc[sig_inj_2, "log2FoldChange"], df.loc[sig_inj_2, "neglog10p"],
               s=20, alpha=0.85, color="orange",    label=f"Higher in inj_2 (n={int(sig_inj_2.sum())})")
    ax.scatter(df.loc[sig_inj_1,  "log2FoldChange"], df.loc[sig_inj_1,  "neglog10p"],
               s=20, alpha=0.85, color="steelblue", label=f"Higher in inj_1 (n={int(sig_inj_1.sum())})")

    ax.axvline(-lfc_thresh, ls="--", lw=0.9, color="black", alpha=0.5)
    ax.axvline( lfc_thresh, ls="--", lw=0.9, color="black", alpha=0.5)
    ax.axhline(-np.log10(pval_thresh), ls="--", lw=0.9, color="black", alpha=0.5)

    for gene, row in df[sig_inj_1].sort_values("pvalue").head(top_n).iterrows():
        ax.text(row["log2FoldChange"], row["neglog10p"], gene, fontsize=7.5, ha="left", va="bottom")
    for gene, row in df[sig_inj_2].sort_values("pvalue").head(top_n).iterrows():
        ax.text(row["log2FoldChange"], row["neglog10p"], gene, fontsize=7.5, ha="right", va="bottom")

    ax.set_xlabel("log₂ Fold Change (inj_1 vs inj_2)")
    ax.set_ylabel("−log₁₀ p-value")
    ax.set_title(f"{ANALYSIS_LABEL} · {phase_name}\ninj_1 vs inj_2  ({note})")
    ax.legend(frameon=False)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


# Choose which method's results to plot:
#   results_wilcoxon  — quick per-cell test
#   results_edgeR     — recommended pseudobulk method
#   results_deseq2    — pseudobulk alternative (if uncommented above)
results = results_edgeR  # ← change as needed

for phase, res in results.items():
    volcano(res, phase_name=phase,
            save_path=f"{OUT_DIR}volcano_{ANALYSIS_LABEL}_{phase}_edgeR.pdf")

### Heatmap of significant DEGs

In [ ]:
for phase, res in results.items():
    sig = res[
        res["pvalue"].notna() &
        (res["pvalue"] < PVALUE_THRESH) &
        (res["log2FoldChange"].abs() > LFC_THRESH)
    ].copy()

    genes = [g for g in sig.index if g in pb_counts.columns]
    if not genes:
        print(f"{phase}: no significant DEGs to plot")
        continue

    samples = pb_meta[pb_meta["phase"] == phase].index
    # log-CPM expression
    expr = np.log1p(pb_counts.loc[samples, genes]).T

    sample_info = pb_meta.loc[samples, [INJURY_COL]].copy()
    sample_info[INJURY_COL] = pd.Categorical(
        sample_info[INJURY_COL].astype(str), categories=["inj_1", "inj_2"], ordered=True
    )
    sample_info = sample_info.sort_values(INJURY_COL)
    expr = expr.loc[:, sample_info.index]
    expr.columns = sample_info[INJURY_COL].astype(str) + "_" + expr.columns.astype(str)

    gene_order = [g for g in sig.sort_values("log2FoldChange").index if g in expr.index]
    expr = expr.loc[gene_order]

    plt.figure(figsize=(8, max(4, len(expr) * 0.25)))
    sns.heatmap(expr, cmap="vlag", xticklabels=True, yticklabels=True)
    n_inj_1 = (sample_info[INJURY_COL] == "inj_1").sum()
    plt.axvline(n_inj_1, color="white", linewidth=2)
    plt.title(f"Significant DEGs — {ANALYSIS_LABEL} — {phase}")
    plt.xlabel("Pseudobulk samples")
    plt.ylabel("Genes")
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}heatmap_DEGs_{ANALYSIS_LABEL}_{phase}.pdf",
                bbox_inches="tight")
    plt.show()

### Summary: gene counts per phase

In [ ]:
summary_rows = []
for phase, res in results.items():
    sig = res[
        res["pvalue"].notna() &
        (res["pvalue"] < PVALUE_THRESH) &
        (res["log2FoldChange"].abs() > LFC_THRESH)
    ]
    summary_rows.append({
        "phase": phase,
        "n_inj_1_up":  int((sig["log2FoldChange"] > 0).sum()),
        "n_inj_2_up": int((sig["log2FoldChange"] < 0).sum()),
        "n_total_sig": len(sig),
        "use_covariate": cov_df.set_index("phase").loc[phase, "use_covariate"],
    })

print(pd.DataFrame(summary_rows).to_string(index=False))